# Equity flows: 13F concentration

## What
Latest top held companies across 13F filers, ranked by reported value, plus the share of the displayed basket in the top 5 names.

## Why this model
A one-period concentration snapshot answers “how top-heavy is reported institutional value?” without inventing a time series the endpoint does not return.

## How to rerun
Needs only `FINUTIES_API_KEY` in `notebooks/.env`. Run top to bottom.

**Endpoint (verified 200):** `GET /api/v1/holdings/top-companies`

**Columns used:** `stock_symbol`, `company_name`, `sector`, `total_value`, `investor_count`

The live schema uses `total_value` and `stock_symbol` — not `market_value` or `symbol`.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


In [2]:
ENDPOINT = "/api/v1/holdings/top-companies"
payload = finuties_get(ENDPOINT, {"limit": 30})
df = pd.DataFrame(normalize_rows(payload) if not isinstance(payload, list) else payload)
if df.empty and isinstance(payload, list):
    df = pd.DataFrame(payload)
require_frame(df, ["stock_symbol", "total_value"], min_rows=5)
assert "market_value" not in df.columns
assert "symbol" not in df.columns

df["total_value"] = pd.to_numeric(df["total_value"], errors="coerce")
df = df.dropna(subset=["stock_symbol", "total_value"])
df = df[df["total_value"] > 0].copy()
require_frame(df, ["stock_symbol", "total_value"], min_rows=5)
require_finite(df["total_value"], "total_value")

top = df.sort_values("total_value", ascending=False).head(15).reset_index(drop=True)
basket = top["total_value"].sum()
assert basket > 0
top["share_pct"] = top["total_value"] / basket * 100
require_finite(top["share_pct"], "share_pct")
concentration_top5 = float(top["share_pct"].head(5).sum())
assert np.isfinite(concentration_top5)

print(f"Top-5 share of shown basket: {concentration_top5:.2f}%")
top[["stock_symbol", "company_name", "sector", "total_value", "share_pct", "investor_count"]]

Top-5 share of shown basket: 59.75%


,stock_symbol,company_name,sector,total_value,share_pct,investor_count
0,MA,NaN,Financial Services,3.548612e+10,19.659447,314
1,CUSIP_67066G104,NVIDIA CORPORATION,NaN,1.969191e+10,10.909392,536
2,GOOGL,ALPHABET INC,Communication Services,1.876553e+10,10.396175,535
3,AAPL,APPLE INC,Technology,1.820815e+10,10.087383,597
4,MSFT,MICROSOFT CORP,Technology,1.570252e+10,8.699258,578
5,AMZN,AMAZON COM INC,Consumer Cyclical,1.276083e+10,7.069552,567
6,GOOG,ALPHABET INC,Communication Services,1.169570e+10,6.479463,483
7,CUSIP_11135F101,BROADCOM INC,NaN,8.536539e+09,4.729276,421
8,CUSIP_30303M102,META PLATFORMS INC,NaN,6.365867e+09,3.526715,457
9,CUSIP_78462F103,STATE STR SPDR S&P 500 ETF T,NaN,6.348308e+09,3.516987,433


## Charts

Left: reported 13F value in USD billions (`total_value`). Right: each name’s share of the displayed top-15 basket, not of the whole market.

In [3]:
labels = top["stock_symbol"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(labels, top["total_value"] / 1e9, color="#1f77b4")
axes[0].set_title("Top holdings by reported value")
axes[0].set_xlabel("Stock symbol")
axes[0].set_ylabel("Reported value (USD billions)")
axes[0].tick_params(axis="x", labelrotation=45)

axes[1].bar(labels, top["share_pct"], color="#9467bd")
axes[1].set_title("Share of displayed top-15 basket")
axes[1].set_xlabel("Stock symbol")
axes[1].set_ylabel("Share (%)")
axes[1].tick_params(axis="x", labelrotation=45)

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
plt.tight_layout()
plt.show()

## Caveats

- 13F values are delayed, long-only, and US-reportable. They are not live fund flows.
- Concentration is computed on the returned top slice, not the full market.
- `company_name` can be null; the chart uses `stock_symbol`.
- Not investment advice.